# MAE, MSE, RMSE — All Three, Side by Side

`30_Cost_Function_MSE_GradientDescent.ipynb` already built MSE from scratch
and ran Gradient Descent on it. `31_Mean_Absolute_Error.MD` and
`32_Root_Mean_Squared_Error.MD` covered MAE and RMSE conceptually, but only
on paper (the 5-student quiz example). This notebook is the hands-on
version: build all three cost functions ourselves, run them on the same
real dataset, and compare what each one reports.

## Step 1 — Load the data

Reusing `placement.csv` (`cgpa` → `package`) — the same file used in
`25`/`26`/`29`/`30` — so every number here is directly comparable to what
came before.

In [1]:
import pandas as pd    # pandas: load the CSV and work with it as a table
import numpy as np     # numpy: arrays, sums, means -- all the math below
import matplotlib.pyplot as plt   # matplotlib: charts, once we get to them

dataset = pd.read_csv("placement.csv")   # same 100-row file used in 25, 26, 29, 30
dataset.head()                           # peek at the first 5 rows -- sanity check

,cgpa,package
0,6.89,3.26
1,5.12,1.98
2,7.82,3.25
3,7.42,3.67
4,6.94,3.57


In [2]:
x = dataset["cgpa"].values      # independent variable, as a plain numpy array
y = dataset["package"].values   # dependent variable (actual values), as a plain numpy array
n = len(x)                      # how many rows/students we have -- the "n" in every formula below
n


100

In [5]:
def predict(x, m, c):
    return m * x + c   # the whole model: multiply slope by x, then shift by intercept

m_bad, c_bad = 0.1, 0.5            # a careless guess -- too flat, shifted up
m_decent, c_decent = 0.59, -1.23   # close to the true best-fit line

y_hat_bad = predict(x, m_bad, c_bad)        # predicted package for every row, using the bad line
y_hat_decent = predict(x, m_decent, c_decent)  # predicted package for every row, using the decent line

y_hat_bad[:5], y_hat_decent[:5]


(array([1.189, 1.012, 1.282, 1.242, 1.194]),
 array([2.8351, 1.7908, 3.3838, 3.1478, 2.8646]))

In [6]:
def mean_squared_error(y_actual, y_predicted):
    errors = y_actual - y_predicted   # actual minus predicted, one number per row
    squared_errors = errors ** 2      # square each one -- negatives become positive, big misses grow fast
    return np.mean(squared_errors)    # average across every row


In [7]:
mean_squared_error(y, y_hat_bad)


np.float64(2.6509399899999995)

In [8]:
def gradients(x, y, m, c):
    y_hat = predict(x, m, c)                     # current line's prediction for every row
    error = y - y_hat                            # how far off each prediction is (signed, not squared)
    dL_dm = -(2 / len(x)) * np.sum(x * error)     # slope of the MSE bowl, in the m direction
    dL_dc = -(2 / len(x)) * np.sum(error)         # slope of the MSE bowl, in the c direction
    return dL_dm, dL_dc


In [9]:
gradients(x, y, m_bad, c_bad)


(np.float64(-21.1708142), np.float64(-3.07718))

In [16]:
learning_rate = 0.02   # step size for each update -- small enough to stay stable, found by testing
epochs = 8000           # how many times we'll nudge m and c

m, c = 0.0, 0.0      # start from the flattest, laziest possible guess
cost_history = []    # track MSE at every step, just so we can watch it fall

for epoch in range(epochs):
    cost_history.append(mean_squared_error(y, predict(x, m, c)))   # how good is the line right now?
    dL_dm, dL_dc = gradients(x, y, m, c)                           # which way is downhill, from here?
    m = m - learning_rate * dL_dm                                  # take one small step downhill
    c = c - learning_rate * dL_dc


m_final, c_final = m, c
m_final, c_final , cost_history[:5], cost_history[-5:]


(np.float64(0.5914952856012292),
 np.float64(-1.2204698463361396),
 [np.float64(7.667213),
  np.float64(5.311518266304031),
  np.float64(3.6914582451691444),
  np.float64(2.577305664311544),
  np.float64(1.8110735085444198)],
 [np.float64(0.09519046377050902),
  np.float64(0.0951904630350011),
  np.float64(0.09519046230049344),
  np.float64(0.09519046156698466),
  np.float64(0.09519046083447331)])